<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/day-26-merge-LoRA-weights-and-perform-before/after%5C-evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U torchao peft transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 17.5 MB/s eta 0:00:00


In [2]:
!pip install -q trl peft transformers datasets accelerate

from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Colab Notebooks/"

import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM = "あなたは会議の議事録から構造化データを抽出するアシスタントです。指定されたJSON形式のみを出力してください。"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from datasets import Dataset

with open(BASE + "pairs.json", encoding="utf-8") as f:
    raw = json.load(f)

formatted = [{
    "prompt": [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": p["note"]},
    ],
    "completion": [
        {"role": "assistant",
         "content": json.dumps(p["json"], ensure_ascii=False)},
    ],
} for p in raw.values()]

dataset = Dataset.from_list(formatted)
split = dataset.train_test_split(test_size=20, seed=42)   # ← seed 要跟 Day 25 一樣
eval_ds = split["test"]
print(len(eval_ds))

20


In [4]:
@torch.no_grad()
def generate(model, messages, max_new_tokens=512):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,                       # greedy，可重現
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)

In [ ]:
from tqdm.notebook import tqdm


base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto"
)






base_model.eval()

before = []
# 使用 tqdm 顯示進度條
for ex in tqdm(eval_ds, desc="正在生成 BEFORE 數據"):
    res = generate(base_model, ex["prompt"])
    before.append(res)

print("\n--- 第一題的輸出範例 ---")
print(before[0][:400])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

正在生成 BEFORE 數據:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
from peft import PeftModel
from tqdm.notebook import tqdm
import torch

# 1. 重新載入原始模型 (Base Model)
ft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2. 掛載你昨天訓練好的 LoRA Adapter
# 請確保 BASE + "lora_adapter_day25" 這個路徑下有你的資料
adapter_path = BASE + "lora_adapter_day25"
print(f"正在從 {adapter_path} 載入 Adapter...")

ft_model = PeftModel.from_pretrained(ft_model, adapter_path)

# 3. 合併權重 (Merge)
print("正在合併模型...")
ft_model = ft_model.merge_and_unload()

# 4. 設定為評估模式
ft_model.eval()

# 5. 開始生成 AFTER 數據
after = []
for ex in tqdm(eval_ds, desc="正在生成 AFTER 數據"):
    res = generate(ft_model, ex["prompt"])
    after.append(res)

print("\n--- 第一題的輸出範例 (AFTER) ---")
print(after[0][:600])